# LangGraph

## Tips

1. Each node should do one thing
2. It will be helpful to restore memory and debugging
3. **STATE**: Before adding a variable into a state, think: should it be shared across the nodes? Will it reduce retrieval (retrieved docs)?

## Notes

- Each node runs when its dependencies are satisfied
- A node with multiple inbound edges is triggered only when **ALL** its required inputs are available

## Workflows: Common Agent Patterns

### Prompt Chaining

Prompt chaining is when each LLM call processes the output of the previous call. It’s often used for performing well-defined tasks that can be broken down into smaller, verifiable steps.
    
### Parallelization

LLMs work simultaneously on a task. This is either done by:
- Running multiple independent subtasks at the same time
- Running the same task multiple times to check for different outputs

### Routing

Based on the state/input, redirect to the correct node.

### Evaluator Optimizer

Generate the answer and verify with a QA LLM model, retry if QA fails.

In [ ]:
### EXAMPLE OF PARALLEL NODE EXECUTION
### example of: Each node runs when its dependencies are satisfied.
### example of: parallel node execution
import os
from typing import Annotated, Optional, TypedDict
from IPython.display import Image
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.graph import START,END, StateGraph
from langgraph.types import Command

# Load API key
load_dotenv()
llm_model = ChatGroq(model_name="openai/gpt-oss-120b", api_key=os.getenv("GROQ_API_KEY"))

# Graph state
class State(TypedDict):
    topic: str
    joke: str
    story: str
    poem: str
    combined_output: str


# Nodes
async def call_llm_1(state: State):
    """First LLM call to generate initial joke"""

    msg = await llm_model.ainvoke(f"Write a joke about {state['topic']}")
    return {"joke": msg.content}


async def call_llm_2(state: State):
    """Second LLM call to generate story"""

    msg = await llm_model.ainvoke(f"Write a story about {state['topic']}")
    return Command(
        update={
            "story": msg.content
        }
    )


async def call_llm_3(state: State):
    """Third LLM call to generate poem"""

    msg = await llm_model.ainvoke(f"Write a poem about {state['topic']}")
    return {"poem": msg.content}


async def aggregator(state: State):
    """Combine the joke, story and poem into a single output"""
    print("Aggregator calling")
    print(state["story"])
    #note it will be only once if it cases multiple edges
    combined = f"Here's a story, joke, and poem about {state['topic']}!\n\n"
    combined += f"STORY:\n{state['story']}\n\n"
    combined += f"JOKE:\n{state['joke']}\n\n"
    combined += f"POEM:\n{state['poem']}"
    msg = await llm_model.ainvoke(f"combine all this content, write blog of it. \n \n {state['topic']}")
    return {"combined_output": msg.content}


# Build workflow
parallel_builder = StateGraph(State)


# Add nodes
parallel_builder.add_node("call_llm_1", call_llm_1)
parallel_builder.add_node("call_llm_2", call_llm_2)
parallel_builder.add_node("call_llm_3", call_llm_3)
parallel_builder.add_node("aggregator", aggregator)

# Add edges to connect nodes
parallel_builder.add_edge(START, "call_llm_1")
parallel_builder.add_edge(START, "call_llm_2")
parallel_builder.add_edge(START, "call_llm_3")
parallel_builder.add_edge("call_llm_1", "aggregator")
parallel_builder.add_edge("call_llm_2", "aggregator")
parallel_builder.add_edge("call_llm_3", "aggregator")
parallel_builder.add_edge("aggregator", END)
parallel_workflow = parallel_builder.compile()

# Show workflow
display(Image(parallel_workflow.get_graph().draw_mermaid_png()))
# Invoke
state = await parallel_workflow.ainvoke({"topic": "cats"})
print(state["combined_output"])

# Persistence & Durability

By default, LangGraph updates the checkpointer after each node execution, but we can change this behavior using durability modes (`exit`/`sync`/`async`).

After each update, graph state will be stored in checkpointer with a unique ID. This will help for resuming or seeing the history of the state updates.

```python
# This is example code to see the state history
# The states are returned in reverse chronological order
states = list(graph.get_state_history(config))

for state in states:
    print(state.next)
    print(state.config["configurable"]["checkpoint_id"])
    print()

# Using the checkpointer ID we can restart the graph at a point with a new state
selected_state = states[1]  # the first checkpoint
new_config = graph.update_state(selected_state.config, values={"topic": "chickens"})
print(new_config)
graph.invoke(None, new_config)  # this will resume the graph with new state
```

## Durability Modes

### Async (Default)

- Checkpoint happens after each node, but in the background
- Graph does **NOT** wait for the checkpoint to finish before continuing
- If a crash occurs right after a node finishes → the last node may re-run because its checkpoint might not be saved yet

```python
graph.invoke(
    {"input": "test"},
    durability="async"  # Default behavior: after node execution but won't wait for next node to start (update happens in background)
)
```

### Sync

- Checkpoint is written before each node starts
- Graph waits for checkpoint write to finish
- Guarantees no node re-runs unless the node itself crashes

```python
graph.stream(
    {"input": "test"},
    durability="sync"  # After each node execution and wait before next node starts
)
```

### Exit

- Checkpoint is written only at the end of the entire graph
- If anything crashes mid-execution → the whole graph starts over

```python
graph.invoke(
    {"input": "test"},
    durability="exit"  # After graph executes
)
```



## Tasks

Tasks come into play when our node executes multiple steps (like doing API calls). Those internal steps can be written into multiple tasks. The task will **not** re-run if the node re-runs because of a crash or interrupt.

```python
from langgraph.func import task

# Side-effect: API call
@task
def fetch_user(user_id: int):
    print("Calling external API...")
    return {"name": "Alice", "id": user_id}

# Side-effect: writing to DB
@task
def save_to_db(data: dict):
    print("Writing to DB...")
    return True

def process_user(state):
    # Step 1: API call (safe with @task)
    user = fetch_user(state["user_id"]).result()

    # Step 2: pure computation (will re-run on replay)
    summary = f"User: {user['name']}"

    # Step 3: DB write (safe with @task)
    save_to_db({"user": user, "summary": summary}).result()

    # Step 4: return output
    return {"summary": summary}
```

# Stream

Stream the agent responses in real-time.

## Stream Modes

- **values**: Streams the full value of the state after each step of the graph
- **updates**: Streams the updates to the state after each step of the graph
- **custom**: Custom stream from nodes (write stream)
- **messages**: LLM output (token-by-token)
- **debug**: Streams as much information as possible throughout the execution of the graph

## Stream Messages Based on LLM Tag

```python
from langchain.chat_models import init_chat_model

# model_1 is tagged with "joke"
model_1 = init_chat_model(model="gpt-4o-mini", tags=['joke'])
# model_2 is tagged with "poem"
model_2 = init_chat_model(model="gpt-4o-mini", tags=['poem'])

graph = ...  # define a graph that uses these LLMs

# The stream_mode is set to "messages" to stream LLM tokens
# The metadata contains information about the LLM invocation, including the tags
async for msg, metadata in graph.astream(
    {"topic": "cats"},
    stream_mode="messages",
):
    # Filter the streamed tokens by the tags field in the metadata to only include
    # the tokens from the LLM invocation with the "joke" tag
    if metadata["tags"] == ["joke"]:
        print(msg.content, end="|", flush=True)
```


In [ ]:
from langgraph.config import get_stream_writer
from langgraph.graph import START, StateGraph
from typing import TypedDict

# Define subgraph
class SubgraphState(TypedDict):
    user: str  # note that this key is shared with the parent graph state
    msg: str
    work: str
    child_work: bool

def subgraph_node_1(state: SubgraphState):
    writer = get_stream_writer()
    writer({"key": "doing operation on node 1 in subgraph"})
    return {"work": state["user"]}

def subgraph_node_2(state: SubgraphState):
    writer = get_stream_writer()
    writer({"key": "doing operation on node 2 in subgraph"})
    writer("done")
    return {"work":state["msg"] + "working as eng", "child_work": True }

subgraph_builder = StateGraph(SubgraphState)
subgraph_builder.add_node(subgraph_node_1)
subgraph_builder.add_node(subgraph_node_2)
subgraph_builder.add_edge(START, "subgraph_node_1")
subgraph_builder.add_edge("subgraph_node_1", "subgraph_node_2")
subgraph = subgraph_builder.compile()

# Define parent graph
class ParentState(TypedDict):
    user: str
    msg: str
    child_work: bool

def node_1(state: ParentState):
    writer = get_stream_writer()
    writer({"key": "doing operation on node_1 in parent graph"})
    return {"msg": "hi " + state["user"]}

builder = StateGraph(ParentState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", subgraph)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
graph = builder.compile()

for chunk in graph.stream(
    {"user": "suresh"},
    stream_mode="debug", # try with values, messages, updates,custom
    # Set subgraphs=True to stream outputs from subgraphs
    subgraphs=True,  
):
    if isinstance(chunk, dict):
        key = chunk.get("key")
        print(key)
    else:
        (path, value) = chunk
        print(chunk)

# Time Travel

LangGraph provides time travel functionality to support these use cases. Specifically, you can resume execution from a prior checkpoint — either replaying the same state or modifying it to explore alternatives. In all cases, resuming past execution produces a new fork in the history.

## Real Use Cases

### Debugging Long/Expensive Workflows Without Re-running Everything

In real projects—RAG agents, multi-step data pipelines, function-calling agents—each step might take:
- Multiple LLM calls
- External API requests
- Tool executions
- Expensive DB queries

If step ★7 fails, without time travel you'd normally have to re-run all the earlier steps. That is slow and wastes tokens.

### Human-in-the-Loop Corrections

Your agent asks the user:
> "Which region should I deploy the cluster?"

User gives an unclear answer → system proceeds → fails later.

With Time Travel, you rewind back to the point where the bad input was given.


## Example Code

```python
# This is example code to see the state history
# The states are returned in reverse chronological order
states = list(graph.get_state_history(config))

for state in states:
    print(state.next)
    print(state.config["configurable"]["checkpoint_id"])
    print()

# Using the checkpointer ID we can restart the graph at a point with a new state
selected_state = states[1]  # the first checkpoint
new_config = graph.update_state(selected_state.config, values={"topic": "chickens"})
print(new_config)
graph.invoke(None, new_config)  # this will resume the graph with new state
```


-----

## Memory

### Types
** Short term memory ** this is check pointer memory used to save the multi-turn conversation

** Long term memory ** store memory, used for accross conversation, user preference, user language, user tone, etc...

```python
    from langgraph.store.memory import InMemoryStore  
    from langgraph.graph import StateGraph
    store = InMemoryStore()  

    builder = StateGraph(...)
    graph = builder.compile(store=store)  

````

For subgraphs, by default parent will store the memory,if you want separation you can create memory for subgraph also

### Sub graph

Two way we can add the subgraph, 

1. Add graph as a node
2. call graph from a node

```python
### type 1
### this apprach, the states are shared, you can access parent state from child state (vice versa)
### state are shared so not completely independent
builder = StateGraph(State)
builder.add_node("node_1", subgraph)  
builder.add_edge(START, "node_1")
graph = builder.compile()


### Type 2
### this is complete independent, we can invoke with manual state
def call_subgraph(state: State):
    # Transform the state to the subgraph state
    subgraph_output = subgraph.invoke({"bar": state["foo"]})  
    # Transform response back to the parent state
    return {"foo": subgraph_output["bar"]}
```
